# Post5G - Add 5G components to a running network

## Prerequisites

Before starting this tutorial, ensure you meet the following requirements:

- Have completed the the [Simple Scenario](./10_simple.ipynb) tutorial.
- Have a basic understanding of Python.
- Are familiar with 5G architecture and 5G NR radio.

## Tutorial Overview

While the [simple scenario](./10_simple.ipynb) deployed a baseline 5G network from scratch, this scenario demonstrates how to modify and extend an existing deployment. 

Specifically, you will expand the network by adding:
* **A Second gNB (`gnb2`):** Implemented using OpenAirInterface (OAI).
* **A Third UE (`sim_ue3`):** Also implemented using OAI, connected to the gNB over an `rfsim` simulated radio channel (physical radio hardware deployments are covered in subsequent scenarios).

The following figure illustrates the augmented network topology.

![Add component topology](./figures/add_component.png)

### UE Configurations

The user equipments are configured with the following parameters:

| UE | SUPI | S-NSSAI | DNN |
| :--- | :--- | :--- | :--- |
| `sim_ue1` | `001010000000100` | `internet` | `internet` |
| `sim_ue2` | `001010000000101` | `internet` | `internet` |
| `sim_ue3` | `001010000000110` | `internet` | `internet` |

## Recovering the Metamodel and Deployment Model of an Existing Deployment

This tutorial assumes a 5G network was previously deployed and configured using the [simple scenario](./10_simple.ipynb) orchestrator workflow. Rather than defining new models from scratch, we can directly retrieve and reuse the metamodel and deployment model from that prior deployment.

In this workflow, the simple scenario saved its state into two files:
* **Metamodel:** `state/simple/metamodel.ues.yaml`
* **Deployment Model:** `state/simple/deployment_model.ues.yaml`

You can load these files directly to initialize your current session's state and begin extending the network.

In [1]:
from metamodel.models.metamodel import MetaModel
from orchestrator.models.deployment import DeploymentModel

metamodel = MetaModel.from_yaml_file('state/simple/metamodel.ues.yaml')
deployment_model = DeploymentModel.from_yaml_file('state/simple/deployment_model.ues.yaml')

We can double check that the models correspond by printing them.

In [2]:
print (metamodel.to_yaml())
print (deployment_model.to_yaml())

ues:
  sim_ue1: {}
  sim_ue2:
    config:
      supi: '001010000000101'

params:
  k8s:
    master_node: node-core
    join_command: 'kubeadm join 10.123.135.0:6443 --token h3as6w.bm1vpjsa06u3vgom
      --discovery-token-ca-cert-hash sha256:b58bd90de90803ff9f714e1277f3e2d36c0a2365296f168b02a9fe3705e69b2d '
    used_ips:
    - 10.244.70.194
    - 10.244.60.53
provisioners:
  my-provisionner:
    type: bi
    params:
      experiment: simple_scenario
      create_experiment: true
      experiment_duration: 1h
      node_duration: 1h
nodes:
  node-core:
    is_provisioned: true
    provisioner: my-provisionner
    ssh_info:
      hostname: 10.123.135.0
      user: dsaucezi
      jumphost:
        hostname: bastion1.theblueprintfactory.org
        user: proxy
        common_args: -o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null -o
          ServerAliveInterval=30 -o ServerAliveCountMax=3 -o ControlMaster=auto -o
          ControlPersist=7200s -o ControlPath=/tmp/cm-%C
      comm

## Adding a gNB

To add a new gNB to the RAN, instantiate a **`RANManagement`** manager using the metamodel and deployment model recovered in the previous step.

In [3]:
from orchestrator.deployment import RANManagement
ran = RANManagement(metamodel=metamodel, deployment_model=deployment_model)

Next, define the metamodel and deployment model for the new gNB.

### Simplified Component Definition via Merged Models

To simplify workflow development, component metamodels and deployment models can be defined simultaneously using a **merged model**. A merged model is simply a dictionary concatenating both the metamodel and deployment model configurations. Internally, orchestrator managers automatically distinguish between metamodel and deployment model keys and reconcile them into the active architecture state.

In this scenario, the merged model simply specifies that the new component is a gNB implemented with OpenAirInterface (OAI). The manager automatically derives all remaining parameters. When no specific Radio Unit (RU) is specified for an OAI gNB, the manager falls back to a simulated `rfsim` radio channel by default. (Examples of advanced configurations and physical radio integrations are covered in subsequent tutorials.)


### Deploying the Component

1. **Add Component:** Call the manager's `add_component()` method, passing the desired name for the new gNB (`gnb2`) along with its merged model.
2. **Execute Deployment:** Trigger the actual deployment across the network by calling the manager's `deploy()` method as usual.

In [4]:
import yaml
merged_model = yaml.safe_load(
"""
type: gnb
implementation: oai
""")

ran.add_component(name="gnb2", merged=merged_model)
await ran.deploy()


PLAY [node-core] ***************************************************************
PLAY [node-core] ***************************************************************


TASK [helm/prepare : Create Helm build directory for 'ueransim-gnb' chart] *****
TASK [helm/prepare : Create Helm build directory for 'oai-gnb2' chart] *********

[WARNING]: Host 'node-core' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
[WARNING]: Host 'node-core' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
changed: [node-core]
ok: [no

By default, OpenAirInterface (OAI) RAN components are deployed in the `oai-ran` namespace within the cluster. 

To verify that the new gNB was deployed successfully, list the active pods in that namespace. You can run the `run_node_command` function using a `ProvisioningManagement` manager instantiated with the updated models from the preceding stage (i.e., `ran`).

In [6]:
from orchestrator.deployment import ProvisioningManagement

prov = ProvisioningManagement(preceding=ran)

print((await prov.run_node_command(
    command=["kubectl", "get", "pods", "-n", "oai-ran", "-o", "wide"],
    nodes=["node-ran"],
    become=True, quiet=True)))

NAME                       READY   STATUS    RESTARTS   AGE   IP             NODE       NOMINATED NODE   READINESS GATES
oai-gnb2-d5fd57bf7-cpfn8   1/1     Running   0          56s   10.244.90.18   node-ran   <none>           <none>


To verify that the new gNB is operating correctly, inspect its container logs by calling the `get_logs()` method on the manager. 

As shown in the output:
* **AMF Connection:** The `amf_ip_address` correctly matches the address of the 5G core deployed in the [simple tutorial](./10_simple.ipynb).
* **NGAP Status:** The NGAP setup procedure successfully completed.
* **Radio Status:** The gNB is actively broadcasting ("on the air").

In [7]:
print(await ran.get_logs(components=["gnb2"], quiet=True))

/proc/sys/kernel/core_pattern=|/usr/share/apport/apport -p%p -s%s -c%c -d%d -P%P -u%u -g%g -F%F -- %E
No configuration file /opt/oai-gnb/etc/gnb.conf found: attempting to find YAML config
== Configuration file:

Active_gNBs:
  - gnb2
# Asn1_verbosity, choice in: none, info, annoying
Asn1_verbosity: none
gNBs:
  ##### Identification parameters:
  - gNB_ID: 0x1

    gNB_name: gnb2

    tracking_area_code: 1

    plmn_list:
      - mcc: 001
        mnc: 01
        mnc_length: 2
        snssaiList:
          - sst: 1

    nr_cellid: 1

    ##### Physical parameters:
    min_rxtxtime: 6

    servingCellConfigCommon:
      #spCellConfigCommon
      - physCellId: 0
        #  downlinkConfigCommon
        #frequencyInfoDL
        # this is 3300.60 MHz + 53*12*30e-3 MHz = 3319.68
        absoluteFrequencySSB: 621312
        # this is 3300.60 MHz
        dl_absoluteFrequencyPointA: 620040
        #scs-SpecificCarrierList
        dl_offstToCarrier: 0
        # subcarrierSpacing
        # 0=kHz15,

## Adding a UE

As the RAN is up and running we can add the new UE. We follow the same approach
as for adding the gNB. As it is a UE component, we instantiate a `UEManagement`
manager.

In [8]:
from orchestrator.deployment import UEManagement
ues = UEManagement(preceding=ran)

In the UE model, specify the desired implementation and its *Subscription Permanent Identifier* (SUPI). Additionally, designate the target gNB for the UE to connect to (in this case, `gnb2`).

You can then add the UE as a new component named `sim_ue3`.

In [9]:
merged_model = yaml.safe_load(
"""
type: containerized
implementation: oai
config:
    supi: '001010000000110'
    gnbs:
    - gnb2
"""
)

ues.add_component(name="sim_ue3", merged=merged_model)

However, you cannot deploy the UE immediately; doing so would cause the 5G core to reject it. Before deploying the UE, its subscriber profile must first be registered in the core database. 

To register the profile, call the `register_ues()` method for this UE using the **`CoreManagement`** manager.

In [10]:
from orchestrator.deployment import CoreManagement

core = CoreManagement(preceding=ues)
await core.register_ues(ues=["sim_ue3"])


PLAY [node-core] ***************************************************************

TASK [database/subscriber/add/open5gs : Add UE 'sim_ue3' to Open5GS database] ***
[WARNING]: Host 'node-core' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
[DEPRECATION WARNING]: Importing 'to_native' from 'ansible.module_utils._text' is deprecated. This feature will be removed from ansible-core version 2.24. Use ansible.module_utils.common.text.converters instead.
changed: [node-core]

PLAY RECAP *********************************************************************
node-core                  : ok=1    changed=1    unreachable=0    failed=0    skipped=0    rescued=0    ignored=0   


Once the UE is registered in the core, you can proceed with its deployment, allowing it to successfully attach and register to the network.

> **Note:** Remember that the orchestrator library strictly follows a declarative, functional paradigm. Always ensure you instantiate new managers with the metamodel and deployment model returned from the most recent deployment stage.

In [11]:
ues = UEManagement(preceding=core)
await ues.deploy(components=["sim_ue3"])


PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-sim-ue3' chart] ******
[WARNING]: Host 'node-core' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
changed: [node-core]

TASK [helm/prepare : Copy custom Helm chart directory for 'oai-sim-ue3' if provided] ***
skipping: [node-core]

TASK [helm/prepare : Copy local built-in 'oai-containerized' Helm chart directory for 'oai-sim-ue3'] ***
changed: [node-core]

TASK [helm/prepare : Render built-in Helm Chart.yaml templated file for 'oai-sim-ue3'] ***
changed: [node-core]

TASK [helm/prepare : Render custom templated Helm files for 'oai-sim-ue3' if provided] ***
skipping: [node-core]

TASK [helm/prepare : Rende

OAI UEs are deployed in the `oai-ue` namespace by default. You can verify that the UE has been deployed successfully by listing the active pods in that namespace.

> **Note 1: Reusing the `ProvisioningManagement` Instance**
> 
> *Why can we use the existing `prov` manager instance instead of instantiating a new `ProvisioningManagement` manager with the latest metamodel and deployment model?*
> 
> Managers only reference the specific sections of the metamodel and deployment model required for their operations. Because subsequent updates to the models do not affect low-level provisioning tasks, you can safely reuse this existing instance without reloading the models.

> **Note 2: Cross-Node Cluster Access**
> 
> *Why can we run `kubectl` commands on the `node-ran` machine even though the UEs are deployed on `node-ues`?*
> 
> The Kubernetes manager provisions a single, unified cluster spanning all infrastructure nodes and automatically installs the `kubeconfig` file on every machine. Consequently, you can execute Kubernetes management commands from any node in the cluster.

In [13]:
print((await prov.run_node_command(
    command=["kubectl", "get", "pods", "-n", "oai-ue", "-o", "wide"],
    nodes=["node-ran"],
    become=True, quiet=True)))

NAME                          READY   STATUS    RESTARTS   AGE   IP             NODE       NOMINATED NODE   READINESS GATES
oai-sim-ue3-548cc87c8-vtb6s   1/1     Running   0          44s   10.244.198.4   node-ues   <none>           <none>


By showing the logs of the UE we can also verify that it has started correctly
and that it received an IP address on its radio interface named `oaitun_ue1`.

In [14]:
print(await ues.get_logs(components=["sim_ue3"], quiet=True))

CMDLINE: "/opt/oai-nr-ue/bin/nr-uesoftmodem" "-O" "/opt/oai-nr-ue/etc/nr-ue.conf" "-E" "--rfsim" 
[UTIL]   running in SA mode (no --phy-test, --do-ra, --nsa option present)
[UTIL]   threadCreate() for Tpool0_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool1_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool2_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool3_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool4_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool5_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool6_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool7_-1: creating thread with affinity ffffffff, priority 97
[OPT]    OPT disabled
[HW]     Version: Branch: develop-c8b78200bf7392b842f7e5b8485837531acaa3e4 Ab

As with the previous UEs, you can verify end-to-end connectivity by pinging `8.8.8.8` from the new UE over its radio interface, confirming that the extended 5G network is fully functional.

In [15]:
command = ["ping", "-I", "oaitun_ue1", "-c", 10, "-D", "8.8.8.8"]
ping = await ues.run_command(command=command, components=["sim_ue3"], quiet=True)
print (ping)

PING 8.8.8.8 (8.8.8.8) from 11.1.1.4 oaitun_ue1: 56(84) bytes of data.
[1792745739.875670] 64 bytes from 8.8.8.8: icmp_seq=1 ttl=114 time=30.6 ms
[1792745740.874614] 64 bytes from 8.8.8.8: icmp_seq=2 ttl=114 time=27.9 ms
[1792745741.875074] 64 bytes from 8.8.8.8: icmp_seq=3 ttl=114 time=26.4 ms
[1792745742.878003] 64 bytes from 8.8.8.8: icmp_seq=4 ttl=114 time=27.9 ms
[1792745743.881170] 64 bytes from 8.8.8.8: icmp_seq=5 ttl=114 time=29.1 ms
[1792745744.881361] 64 bytes from 8.8.8.8: icmp_seq=6 ttl=114 time=28.7 ms
[1792745745.883579] 64 bytes from 8.8.8.8: icmp_seq=7 ttl=114 time=29.6 ms
[1792745746.881995] 64 bytes from 8.8.8.8: icmp_seq=8 ttl=114 time=26.3 ms
[1792745747.883513] 64 bytes from 8.8.8.8: icmp_seq=9 ttl=114 time=26.5 ms
[1792745748.885842] 64 bytes from 8.8.8.8: icmp_seq=10 ttl=114 time=27.6 ms

--- 8.8.8.8 ping statistics ---
10 packets transmitted, 10 received, 0% packet loss, time 9013ms
rtt min/avg/max/mdev = 26.337/28.043/30.562/1.365 ms



> **Note:** You may notice that the network latency is significantly higher here than when pinging from UERANSIM UEs. This difference occurs because UERANSIM bypasses protocol-stack radio processing, whereas OAI gNBs and `rfsim` fully implement and simulate the physical radio layer. Consequently, the measured latencies accurately reflect real-world resource block allocations and TDD configurations.

To further validate the correctness of the 5G network, you can test inter-UE communication by pinging another registered UE from your newly added UE. For example, let's ping `sim_ue1`.

You can retrieve the IP address of `sim_ue1` by running the `ip addr show` command directly inside its container, or by referencing the UE logs extracted during the [simple tutorial](./simple.ipynb).

In [18]:
command = ["bash", "-c", "'ip -4 addr show uesimtun0 | grep -oP \"(?<=inet )\\d+(\\.\\d+){3}\"'"]
sim_ue1_ip = (await ues.run_command(command=command, components=["sim_ue1"], quiet=True)).rstrip()
print (f"IP address of sim_ue1 {sim_ue1_ip}")

IP address of sim_ue1 11.1.1.2


In [ ]:
command = ["ping", "-I", "oaitun_ue1", "-c", 10, "-D", sim_ue1_ip]
ping = await ues.run_command(command=command, components=["sim_ue3"], quiet=True)

In [20]:
print (ping)

PING 11.1.1.2 (11.1.1.2) from 11.1.1.4 oaitun_ue1: 56(84) bytes of data.
[1792745839.431338] 64 bytes from 11.1.1.2: icmp_seq=1 ttl=64 time=24.1 ms
[1792745840.429798] 64 bytes from 11.1.1.2: icmp_seq=2 ttl=64 time=21.4 ms
[1792745841.434648] 64 bytes from 11.1.1.2: icmp_seq=3 ttl=64 time=24.8 ms
[1792745842.436525] 64 bytes from 11.1.1.2: icmp_seq=4 ttl=64 time=24.9 ms
[1792745843.440296] 64 bytes from 11.1.1.2: icmp_seq=5 ttl=64 time=26.8 ms
[1792745844.438563] 64 bytes from 11.1.1.2: icmp_seq=6 ttl=64 time=23.2 ms
[1792745845.440504] 64 bytes from 11.1.1.2: icmp_seq=7 ttl=64 time=24.1 ms
[1792745846.440690] 64 bytes from 11.1.1.2: icmp_seq=8 ttl=64 time=23.1 ms
[1792745847.440986] 64 bytes from 11.1.1.2: icmp_seq=9 ttl=64 time=22.3 ms
[1792745848.441917] 64 bytes from 11.1.1.2: icmp_seq=10 ttl=64 time=21.9 ms

--- 11.1.1.2 ping statistics ---
10 packets transmitted, 10 received, 0% packet loss, time 9013ms
rtt min/avg/max/mdev = 21.409/23.652/26.772/1.535 ms



Finally, we can archive the metamodel and deployment model in file for keeping
track of the experiment models.

In [21]:
from pathlib import Path
DIR="state/add_components"
Path(DIR).mkdir(parents=True, exist_ok=True)

ues.metamodel.to_yaml_file(f"{DIR}/metamodel.ues.yaml")
ues.deployment_model.to_yaml_file(f"{DIR}/deployment_model.ues.yaml")